In [0]:
# Install required packages
%pip install langchain-core langchain-community databricks-langchain -q
dbutils.library.restartPython()

In [0]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_community.chat_models import ChatDatabricks

# Initialize the LLM using Databricks Foundation Model
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.7,
    max_tokens=500
)

# Create a PromptTemplate for product description generation
product_template = """
You are a creative marketing copywriter. Generate compelling product content for the following product.

Product Name: {product_name}

Please provide:
1. A detailed and engaging product description (2-3 sentences)
2. A list of 5 key features (bullet points)
3. A catchy marketing tagline

Format your response as:

**Product Description:**
[description here]

**Key Features:**
- [feature 1]
- [feature 2]
- [feature 3]
- [feature 4]
- [feature 5]

**Marketing Tagline:**
[tagline here]
"""

prompt = PromptTemplate(
    input_variables=["product_name"],
    template=product_template
)

# Create a Chain using the pipe operator (LCEL - LangChain Expression Language)
product_chain = prompt | llm

# Example: Generate content for "Wireless Earbuds"
product_name = "Wireless Earbuds"
result = product_chain.invoke({"product_name": product_name})

print(f"\n{'='*60}")
print(f"PRODUCT CONTENT FOR: {product_name}")
print(f"{'='*60}\n")
print(result.content)

In [0]:
# Try with different products
products = [
    "Smart Water Bottle",
    "Ergonomic Office Chair",
    "Portable Solar Charger"
]

for product in products:
    result = product_chain.invoke({"product_name": product})
    print(f"\n{'='*60}")
    print(f"PRODUCT: {product}")
    print(f"{'='*60}\n")
    print(result.content)
    print("\n")

In [0]:
# Alternative: Use Databricks widgets for a cleaner UI
# This creates a text input widget at the top of the notebook

# Remove existing widget if it exists
dbutils.widgets.removeAll()

# Create a text input widget
dbutils.widgets.text("product_name", "Smart Watch", "Enter Product Name:")

# Get the value from the widget
user_product = dbutils.widgets.get("product_name")

if user_product.strip():
    print(f"Generating content for: {user_product}...\n")
    
    result = product_chain.invoke({"product_name": user_product})
    
    print(f"\n{'='*60}")
    print(f"PRODUCT: {user_product}")
    print(f"{'='*60}\n")
    print(result.content)
else:
    print("Please enter a product name in the widget above and re-run this cell.")